# Build final_dataset.csv

Reads the three pipeline output CSVs in this folder, adds a **canonical_solution** column (from the original datasets), and saves one combined **final_dataset.csv** with all rows and canonical_solution as the last column.

Runs locally; uses cached HuggingFace datasets for canonical/reference solutions (downloads once if needed).

In [1]:
import pandas as pd
import os

# Folder containing the three pipeline output CSVs (run notebook from AST+DYNMAIC+LIB_API or set FOLDER)
FOLDER = os.getcwd()
print("Using folder:", FOLDER)

Using folder: d:\Desktop\MIT\CODES\FOR GIT HUB\FYP-26\Pipeline construction\AST+DYNMAIC+LIB_API


In [2]:
# Optional: set FOLDER to the path containing ds1000/humaneval/mbpp_pipeline_output.csv
# If not set, the folder where the notebook is run (getcwd()) is used.
if not os.path.isfile(os.path.join(FOLDER, "ds1000_pipeline_output.csv")):
    alt = os.path.join(os.getcwd(), "Pipeline construction", "AST+DYNMAIC+LIB_API")
    if os.path.isfile(os.path.join(alt, "ds1000_pipeline_output.csv")):
        FOLDER = alt
        print("Using alternate folder:", FOLDER)

## 1. Load canonical/reference solutions from datasets

Maps `task_id` to canonical solution text (reference_code for DS1000, canonical_solution for HumanEval, code for MBPP). First run may download datasets; after that uses cache.

In [3]:
from datasets import load_dataset

def get_ds1000_canonical():
    ds = load_dataset("xlangai/DS-1000")
    df = ds["test"].to_pandas()
    df["task_id"] = [f"DS{str(i).zfill(4)}" for i in range(len(df))]
    return df[["task_id", "reference_code"]].rename(columns={"reference_code": "canonical_solution"})

def get_humaneval_canonical():
    ds = load_dataset("openai/openai_humaneval")
    df = ds["test"].to_pandas()
    return df[["task_id", "canonical_solution"]]

def get_mbpp_canonical():
    ds = load_dataset("google-research-datasets/mbpp", "sanitized")
    df = ds["train"].to_pandas()
    df["task_id"] = df["task_id"].astype(str)
    return df[["task_id", "code"]].rename(columns={"code": "canonical_solution"})

print("Loading canonical mappings (may download on first run)...")
canon_ds1000 = get_ds1000_canonical()
canon_humaneval = get_humaneval_canonical()
canon_mbpp = get_mbpp_canonical()
print("DS1000:", len(canon_ds1000), "rows")
print("HumanEval:", len(canon_humaneval), "rows")
print("MBPP:", len(canon_mbpp), "rows")

Loading canonical mappings (may download on first run)...


DS1000: 1000 rows
HumanEval: 164 rows
MBPP: 120 rows


## 2. Load pipeline outputs and attach canonical_solution

In [4]:
paths = {
    "ds1000": os.path.join(FOLDER, "ds1000_pipeline_output.csv"),
    "humaneval": os.path.join(FOLDER, "humaneval_pipeline_output.csv"),
    "mbpp": os.path.join(FOLDER, "mbpp_pipeline_output.csv"),
}

canon_map = {
    "ds1000": canon_ds1000,
    "humaneval": canon_humaneval,
    "mbpp": canon_mbpp,
}

frames = []
for name, path in paths.items():
    if not os.path.isfile(path):
        print(f"Skip {name}: file not found {path}")
        continue
    df = pd.read_csv(path)
    df["task_id"] = df["task_id"].astype(str)
    canon = canon_map[name]
    # Merge on task_id to add canonical_solution
    df = df.merge(canon, on="task_id", how="left")
    df["canonical_solution"] = df["canonical_solution"].fillna("")
    frames.append(df)
    print(f"{name}: {len(df)} rows")

if not frames:
    raise SystemExit("No pipeline CSV files found.")

ds1000: 1000 rows
humaneval: 164 rows
mbpp: 327 rows


## 3. Combine into one dataframe and ensure canonical_solution is last

In [5]:
combined = pd.concat(frames, ignore_index=True)

# Ensure canonical_solution exists and is last column
if "canonical_solution" not in combined.columns:
    combined["canonical_solution"] = ""
cols = [c for c in combined.columns if c != "canonical_solution"] + ["canonical_solution"]
combined = combined[cols]

out_path = os.path.join(FOLDER, "final_dataset.csv")
combined.to_csv(out_path, index=False)
print(f"Saved {len(combined)} rows to {out_path}")
print("Columns:", list(combined.columns))

Saved 1491 rows to d:\Desktop\MIT\CODES\FOR GIT HUB\FYP-26\Pipeline construction\AST+DYNMAIC+LIB_API\final_dataset.csv
Columns: ['dataset', 'task_id', 'status', 'ast_info', 'dynamic_info', 'lib_info', 'generated_code', 'patched_code', 'error_sources', 'error_types', 'error_lines', 'canonical_solution']
